# Notebook 41 — confusion-adaptive proxy-margin Stage A

This is a single-candidate extension of `40_balanced_supcon.ipynb`. The deployed architecture remains the `47 → 128 → 64` shared encoder, four linear `64 → 22` experts, and frozen-body `64 → 32 → 4` Stage-C gate. Stage A replaces balanced SupCon with weighted CE plus a `0.10` confusion-adaptive proxy-margin term. The confusion state is learned only from the current run's training predictions, uses absolute off-diagonal leakage, and is discarded with the temporary Stage-A head.

There is no uniform-margin, shuffled-matrix, calibration-split, or label-conflict condition in this notebook. Therefore it evaluates the complete method against Notebook 40, not the isolated causal effect of the matrix.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
REFERENCE_PREFIX = 'nfv3_4way_moe_soft_balanced_supcon_deeper_gate32_frozen_stagec_v1'
REFERENCE_CONFIG = 'config/compact_soft_moe_balanced_supcon_frozen_deeper_gate_3seed.yaml'
CANDIDATE_PREFIX = 'nfv3_4way_moe_soft_confusion_adaptive_margin_deeper_gate32_frozen_stagec_v1'
CANDIDATE_CONFIG = 'config/compact_soft_moe_confusion_adaptive_margin_frozen_deeper_gate_3seed.yaml'
EXECUTE = False  # Run the dry-run first; then change to True.
MAX_NEW_SEEDS = 2  # Initial paired seeds 0–1; use 1 later to confirm seed 2.
RUN_TESTS = True
# ==================================================================

## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token: raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}'}
repo = Path('/content') / GITHUB_REPO; url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir(): subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else: subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None; os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR; os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'; os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Contract preflight and candidate execution

The preflight removes the complete Stage-A representation blocks and requires every remaining architecture and training field to match Notebook 40. The candidate retrains Stages A and B because its encoder coordinates change; Stage C remains gate-only.

In [ ]:
import copy, torch, yaml
from models.encoder import build_encoder
from models.representation_losses import StageARepresentationObjective
from training.logging_utils import run_streaming_logged
from training.model_utils import apply_stage_c_trainability, build_model
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime before executing the study.')
reference = yaml.safe_load(Path(REFERENCE_CONFIG).read_text())
candidate = yaml.safe_load(Path(CANDIDATE_CONFIG).read_text())
assert reference['seeds'] == candidate['seeds'] == [0, 1, 2]
assert 'reuse_stage_a_from_prefix' not in candidate and 'reuse_stage_b_from_prefix' not in candidate
ref = copy.deepcopy(reference['backbone']); cand = copy.deepcopy(candidate['backbone'])
ref_representation = ref['training'].pop('representation'); cand_representation = cand['training'].pop('representation')
assert ref == cand, 'Notebook 41 may change only the Stage-A representation method'
assert ref_representation['objective'] == 'balanced_supcon'
assert cand_representation['objective'] == 'confusion_adaptive_margin'
for key, value in ref_representation.items():
    if key != 'objective': assert cand_representation[key] == value
expected_adaptive = {'weight': 0.1, 'temperature': 0.1, 'confusion_warmup_epochs': 5, 'confusion_ema': 0.5, 'confusion_shrinkage': 100.0, 'confusion_top_k': 2, 'confusion_max_margin': 0.15}
for key, value in expected_adaptive.items(): assert cand_representation[key] == value
assert cand_representation['sampling'] == 'legacy' and cand_representation['class_weighting'] == 'legacy'
assert not any('matrix' in key or 'calibration' in key for key in cand_representation), 'No external matrix/calibration input is allowed'
model_cfg = candidate['backbone']['model']
encoder = build_encoder(47, 64, model_cfg['encoder'])
model = build_model(candidate['backbone']['architecture'], encoder, ['UNSW', 'ToN', 'BoT', 'CIC'], [f'class_{i}' for i in range(22)], model_cfg)
apply_stage_c_trainability(model, candidate['backbone'])
assert sum(p.numel() for p in model.parameters()) == candidate['expected_total_parameters'] == 22332
assert sum(p.numel() for p in model.parameters() if p.requires_grad) == 2212
objective = StageARepresentationObjective(64, 22, candidate['backbone'])
assert objective.objective == 'confusion_adaptive_margin'
if RUN_TESTS: subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_representation_losses.py', 'tests/test_compact_soft_moe_run.py', 'tests/test_notebook31_ablation_methods.py', 'tests/test_dataset_blind_inference.py', 'tests/test_latent_space.py', 'tests/test_out_of_core.py'], check=True)
summary_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{CANDIDATE_PREFIX}_summary'; summary_dir.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-u', '-m', 'training.compact_soft_moe_run', '--study-config', CANDIDATE_CONFIG, '--prefix', CANDIDATE_PREFIX]
run_streaming_logged(command, str(summary_dir / 'Notebook_Dry_Run.log'), env=os.environ.copy(), heartbeat_seconds=30, label='41-confusion-margin-dry-run')
if EXECUTE: run_streaming_logged([*command, '--execute', '--max-new-seeds', str(MAX_NEW_SEEDS)], str(summary_dir / 'Notebook_Training_Stream.log'), env=os.environ.copy(), heartbeat_seconds=30, label='41-confusion-margin-training')
else: print('NO TRAINING WAS STARTED. Review the contract, set EXECUTE=True, and rerun this cell.')

## Common paired seeds: validation first

Only seeds completed by both Notebook 40 Balanced SupCon and Notebook 41 enter a comparison. With one common seed the output is explicitly exploratory; a method-level conclusion requires at least seeds 0–1. Locked-test metrics are descriptive and never select the method.

In [ ]:
import numpy as np, pandas as pd
from IPython.display import Image, display
conditions = {'balanced_supcon': REFERENCE_PREFIX, 'confusion_adaptive_margin': CANDIDATE_PREFIX}
validation_rows = []; test_rows = []; class_rows = []; dataset_rows = []; resource_rows = []
for condition, prefix in conditions.items():
    for seed in (0, 1, 2):
        result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_seed{seed}'
        required = ['manifest.json', 'Validation_Overall_Metrics.csv', 'Validation_Per_Dataset_Metrics.csv', 'Validation_Per_Class_Metrics.csv', 'Overall_Metrics.csv', 'Per_Class_Metrics.csv', 'Per_Dataset_Metrics.csv', 'Resource_Accounting.csv']
        if not all((result_dir / name).is_file() for name in required): continue
        overall_val = pd.read_csv(result_dir / 'Validation_Overall_Metrics.csv').iloc[0]
        val_ds = pd.read_csv(result_dir / 'Validation_Per_Dataset_Metrics.csv')
        val_cls = pd.read_csv(result_dir / 'Validation_Per_Class_Metrics.csv')
        present = val_cls['support'] > 0; rare = val_cls[(val_cls['support'] >= 20) & (val_cls['support'] < 200)]
        weighted_f1 = np.average(val_cls.loc[present, 'f1'], weights=val_cls.loc[present, 'support'])
        validation_rows.append({'condition': condition, 'seed': seed, 'validation_macro_f1': overall_val['macro_f1'], 'validation_weighted_f1': weighted_f1, 'validation_accuracy': overall_val['accuracy'], 'validation_macro_precision': overall_val['macro_precision'], 'validation_macro_recall': overall_val['macro_recall'], 'worst_dataset_macro_f1': val_ds['macro_f1'].min(), 'rare_recall_mean': rare['recall'].mean() if len(rare) else np.nan, 'weakest_class_recall': val_cls.loc[present, 'recall'].min()})
        overall_test = pd.read_csv(result_dir / 'Overall_Metrics.csv').query("origin == 'ALL'").copy(); overall_test.insert(0, 'seed', seed); overall_test.insert(0, 'condition', condition); test_rows.append(overall_test)
        per_class = pd.read_csv(result_dir / 'Per_Class_Metrics.csv').query("origin == 'ALL'").copy(); per_class.insert(0, 'seed', seed); per_class.insert(0, 'condition', condition); class_rows.append(per_class)
        per_dataset = pd.read_csv(result_dir / 'Per_Dataset_Metrics.csv').copy(); per_dataset.insert(0, 'seed', seed); per_dataset.insert(0, 'condition', condition); dataset_rows.append(per_dataset)
        resources = pd.read_csv(result_dir / 'Resource_Accounting.csv').copy(); resources['seed'] = seed; resources['condition'] = condition; resource_rows.append(resources)
validation = pd.DataFrame(validation_rows)
if validation.empty: print('No completed reference/candidate results exist yet.')
else:
    common = sorted(set(validation.query("condition == 'balanced_supcon'")['seed']) & set(validation.query("condition == 'confusion_adaptive_margin'")['seed']))
    print('Completed common paired seeds:', common)
    if len(common) < 2: print('EXPLORATORY ONLY: fewer than two common paired seeds; do not claim a method-level average.')
    paired_validation = validation[validation['seed'].isin(common)].copy(); display(paired_validation)
    metrics = ['validation_macro_f1', 'validation_weighted_f1', 'validation_accuracy', 'validation_macro_precision', 'validation_macro_recall', 'worst_dataset_macro_f1', 'rare_recall_mean', 'weakest_class_recall']
    for metric in metrics:
        pivot = paired_validation.pivot(index='seed', columns='condition', values=metric)
        pivot['paired_delta_confusion_minus_bsupcon'] = pivot['confusion_adaptive_margin'] - pivot['balanced_supcon']
        print(metric); display(pivot); print('Mean paired delta:', pivot['paired_delta_confusion_minus_bsupcon'].mean(), 'sample SD:', pivot['paired_delta_confusion_minus_bsupcon'].std(ddof=1))
    if common and test_rows:
        test = pd.concat(test_rows, ignore_index=True).query('seed in @common')
        test_metrics = [c for c in ['accuracy', 'micro_f1', 'weighted_f1', 'macro_f1', 'macro_precision', 'macro_recall', 'roc_auc_ovr_macro', 'pr_auc_ovr_macro'] if c in test]
        print('=== Locked-test descriptive metrics; never used for selection ==='); display(test.groupby('condition')[test_metrics].agg(['mean', 'std']))
        print('=== Per-dataset test macro-F1 ==='); display(pd.concat(dataset_rows, ignore_index=True).query('seed in @common').pivot_table(index='dataset', columns='condition', values='macro_f1', aggfunc='mean'))
        print('=== Per-class test F1 ==='); display(pd.concat(class_rows, ignore_index=True).query('seed in @common').pivot_table(index='class', columns='condition', values='f1', aggfunc='mean'))
    if resource_rows:
        resources = pd.concat(resource_rows, ignore_index=True); display(resources)
        candidate_resources = resources.query("condition == 'confusion_adaptive_margin'")
        assert set(candidate_resources['total_parameters'].astype(int)) == {22332}
        assert set(candidate_resources['trainable_parameters'].astype(int)) == {2212}

## Adaptive-matrix, Stage-B transfer, and frozen Stage-C diagnostics

In [ ]:
for seed in (0, 1, 2):
    result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{CANDIDATE_PREFIX}_seed{seed}'
    if not (result_dir / 'manifest.json').is_file(): continue
    matrices = pd.read_csv(result_dir / 'Confusion_Adaptive_Matrices.csv')
    rivals = pd.read_csv(result_dir / 'Confusion_Adaptive_Rivals.csv')
    adaptive = pd.read_csv(result_dir / 'Confusion_Adaptive_Epoch_Summary.csv')
    raw = matrices.query("matrix == 'raw'")
    margins = matrices.query("matrix == 'margins'")
    assert np.allclose(raw.query('true_class == rival_class')['value'], 0.0)
    assert margins['value'].max() <= 0.15 + 1e-8
    nonzero = margins.query('value > 0').groupby(['epoch', 'true_class']).size()
    assert (nonzero <= 2).all()
    print(f'=== Seed {seed}: adaptive epoch summary ==='); display(adaptive)
    print(f'=== Seed {seed}: final selected rivals ==='); display(rivals[rivals['epoch'] == rivals['epoch'].max()])
    reference_b = pd.read_csv(Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / f'{REFERENCE_PREFIX}_seed{seed}' / 'Validation_Expert_Owned_StageB.csv')
    candidate_b = pd.read_csv(Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / f'{CANDIDATE_PREFIX}_seed{seed}' / 'Validation_Expert_Owned_StageB.csv')
    stage_b = reference_b.merge(candidate_b, on='dataset', suffixes=('_bsupcon', '_confusion'))
    for metric in ('macro_f1', 'macro_precision', 'macro_recall', 'accuracy'): stage_b[f'delta_{metric}'] = stage_b[f'{metric}_confusion'] - stage_b[f'{metric}_bsupcon']
    print(f'=== Seed {seed}: Stage-B transfer versus Balanced SupCon ==='); display(stage_b)
    candidate_c = pd.read_csv(result_dir / 'Validation_Expert_Owned_StageC.csv')
    preserved = candidate_b.merge(candidate_c, on='dataset', suffixes=('_stage_b', '_stage_c')); delta_columns = []
    for metric in ('macro_f1', 'macro_precision', 'macro_recall', 'accuracy'):
        column = f'delta_{metric}'; preserved[column] = preserved[f'{metric}_stage_c'] - preserved[f'{metric}_stage_b']; delta_columns.append(column)
    assert np.allclose(preserved[delta_columns], 0.0, atol=1e-12), f'Frozen-body drift in seed {seed}'
    print(f'=== Seed {seed}: exact Stage-B competence preservation ==='); display(preserved)
    print(f'=== Seed {seed}: gate entropy/confidence by dataset ==='); display(pd.read_csv(result_dir / 'Gate_By_Dataset.csv'))
    print(f'=== Seed {seed}: expert utilization ==='); display(pd.read_csv(result_dir / 'Expert_Utilization.csv'))

## Training and latent-space diagnostics

In [ ]:
latent_snapshots = []; latent_probes = []
available_common = common if 'common' in globals() else []
for condition, prefix in conditions.items():
    for seed in available_common:
        result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_seed{seed}'
        cumulative = result_dir / 'latent' / 'cumulative'
        for filename, target in [('Latent_Snapshot_Metrics.csv', latent_snapshots), ('Latent_Probe_Metrics.csv', latent_probes)]:
            path = cumulative / filename
            if path.is_file():
                frame = pd.read_csv(path); frame.insert(0, 'seed', seed); frame.insert(0, 'condition', condition); target.append(frame)
for seed in (0, 1, 2):
    result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{CANDIDATE_PREFIX}_seed{seed}'
    history_path = result_dir / 'training' / 'Training_History.csv'
    if not history_path.is_file(): continue
    history = pd.read_csv(history_path)
    stage_a_columns = [c for c in ['epoch', 'objective', 'train_ce_loss', 'train_representation_loss', 'train_weighted_representation_loss', 'train_representation_fraction', 'train_total_loss'] if c in history]
    print(f'=== Seed {seed}: Stage A ==='); display(history.query("stage == 'A'")[stage_a_columns])
    print(f'=== Seed {seed}: Stage C ==='); display(history.query("stage == 'C'")[[c for c in ['epoch', 'learning_rate', 'train_total_loss', 'train_ce_loss', 'train_balance_penalty', 'val_macro_f1', 'best_val_macro_f1', 'patience_left'] if c in history]])
if latent_snapshots:
    latent_snapshots = pd.concat(latent_snapshots, ignore_index=True)
    print('=== Paired latent geometry ==='); display(latent_snapshots.groupby(['condition', 'snapshot'])[[c for c in ['silhouette', 'effective_rank'] if c in latent_snapshots]].agg(['mean', 'std']))
if latent_probes:
    latent_probes = pd.concat(latent_probes, ignore_index=True)
    print('=== Paired latent probes ==='); display(latent_probes.groupby(['condition', 'snapshot', 'probe', 'target'])[[c for c in ['macro_f1_present', 'accuracy'] if c in latent_probes]].agg(['mean', 'std']))
candidate_summary = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{CANDIDATE_PREFIX}_summary'
if (candidate_summary / 'Final_Study_Manifest.json').is_file():
    for figure in sorted((candidate_summary / 'training_figures').glob('*.png')): display(Image(filename=str(figure)))

## Interpretation rule

Select the complete confusion-adaptive method only from paired validation results. With fewer than two common seeds, treat the result as exploratory. If most learned margin mass targets Benign, describe the mechanism as strengthened attack-versus-Benign separation rather than broad rival-pair learning. The matrices may also contain irreducible label ambiguity because this experiment deliberately does not add a duplicate/near-duplicate conflict correction.